# 04 Train Probability Model

Train an Elo-only probability baseline and a logistic regression model on engineered features.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mlops.data import load_matches
from mlops.features import build_match_features
from mlops.modeling import elo_probability_from_diff, train_logistic_regression
from mlops.splits import chronological_split, split_features_and_target

DATA_PATH = ROOT / "data" / "matchs_stats.csv"
df = load_matches(DATA_PATH)
feature_df = build_match_features(df)
labeled_df = feature_df[feature_df["blue_team_win"].notna()].copy()
labeled_df["blue_team_win"] = labeled_df["blue_team_win"].astype("Int64")
train_df, valid_df, test_df = chronological_split(labeled_df)
X_train, y_train = split_features_and_target(train_df)
X_valid, y_valid = split_features_and_target(valid_df)

elo_valid_prob = elo_probability_from_diff(valid_df["elo_diff"])
model = train_logistic_regression(X_train, y_train)
logreg_valid_prob = pd.Series(model.predict_proba(X_valid)[:, 1], index=valid_df.index)
pd.DataFrame(
    {
        "elo_valid_prob": elo_valid_prob,
        "logreg_valid_prob": logreg_valid_prob,
        "blue_team_win": y_valid,
    }
).head()

,elo_valid_prob,logreg_valid_prob,blue_team_win
0,0.439412,0.536574,1
1,0.562208,0.654475,0
2,0.500000,0.605720,0
3,0.396579,0.484469,1
4,0.566676,0.641637,1


In [2]:
pd.DataFrame(
    {
        "elo_valid_prob": elo_valid_prob.describe(),
        "logreg_valid_prob": logreg_valid_prob.describe(),
    }
)

,elo_valid_prob,logreg_valid_prob
count,160.000000,160.000000
mean,0.500649,0.566999
std,0.114750,0.134562
min,0.201883,0.207041
25%,0.412328,0.470438
50%,0.502104,0.573408
75%,0.581775,0.678861
max,0.799368,0.850297
